# Back massage - whole back

Your back is 3 rows (top, middle, bottom) × left/right. The arm massages each row, slides left↔right inside a row, and changes rows **only through the air**.

**Every time:** close other arm notebooks → Kernel → Restart Kernel and Clear Outputs → run cell 1 and cell 2.

**■ Stop** = lifts off and sleeps. **Arm power plug** = emergency stop.

## 1. Settings

In [ ]:
import time, math, random, io, contextlib

SIM = False              # True = test on the laptop without moving the arm

# ---- YOUR BACK -------------------------------------------------------------
# Your 6 readings: where the gripper touches your back. Rows go from top (shoulders) to bottom.
# "air" = the arm lifted AWAY above that row (hold the arm, lift it 5-10 cm straight up from
# the row, read the numbers). The arm moves between rows ONLY through these air poses.
ROWS = [
    {"name": "top",
     "left":  [179, 84, 103, 99, 89, 150],
     "right": [179, 62, 105, 157, 89, 150],
     "air":   None},          # <- record and fill in, e.g. [179, 60, 90, 120, 89, 150]
    {"name": "middle",
     "left":  [15, 97, 50, 76, 95, 150],
     "right": [21, 114, 72, 22, 96, 150],
     "air":   None},          # <- record and fill in
    {"name": "bottom",
     "left":  [17, 97, 74, 62, 96, 150],
     "right": [47, 84, 58, 86, 96, 150],
     "air":   None},          # <- record and fill in
]
PATH_CHECKED = False     # set True after "Step 1: check the path" looked safe. Scratch won't run before.

SLEEP_POSE = None        # None = rest in the top row's air pose. Or your own numbers.

# ---- FEEL --------------------------------------------------------------------
MINUTES = 10             # then it lifts off and goes to sleep
MAX_SPEED = 25           # degrees per second for any motor. Lower = slower, smoother, quieter (15-40)
STRENGTH = 1.0           # size of the scratch/massage moves. 0.5 = small/gentle, 1.5 = bigger
SOFT = 0.0               # 0 = touch exactly at your points. 0.05-0.1 = lighter touch (a little toward air)
GRIP_OPEN = 105
GRIP_CLOSED = 150
MODE = "mix"             # "mix", "massage" or "scratch"

# Safety: while touching you, no motor may go further than this outside the row's recorded poses
MAX_FROM_POSE = 12


## 2. Program + connect (just run it)

In [ ]:
STEP_MS = 40   # one small step every 40 ms -> smooth, calligraphy-like
_LIMITS = {1: 180, 2: 180, 3: 180, 4: 180, 5: 270, 6: 180}

class SimArm:
    def __init__(self):
        self.a = list(ROWS[0]["left"])
        self.log = []
    def Arm_serial_servo_write6_array(self, a, t):
        self.a = list(a); self.log.append(list(a))
    def Arm_serial_servo_read(self, s):
        return self.a[s - 1]
    def Arm_serial_set_torque(self, on):
        pass

def connect():
    if SIM:
        return SimArm()
    from Arm_Lib import Arm_Device
    a = Arm_Device()
    time.sleep(0.1)
    return a

def mix(a, b, f):
    return [x + (y - x) * f for x, y in zip(a, b)]

AIR_READY = all(r.get("air") for r in ROWS)

def air(r):
    return list(ROWS[r]["air"])

def spot(r, side):
    """side: 0 = left, 1 = right, anything between = in between."""
    return mix(ROWS[r]["left"], ROWS[r]["right"], side)

def toward_air(pose, r, f):
    """Move pose a fraction f of the way toward the row's air pose = away from your back."""
    return mix(pose, air(r), max(0.0, f))

def touch(r, side):
    return toward_air(spot(r, side), r, SOFT) if AIR_READY else spot(r, side)

def hover(r, side):
    return toward_air(spot(r, side), r, 0.35)

SLEEP = list(SLEEP_POSE) if SLEEP_POSE else (air(0) if AIR_READY else None)

_cur = None
_guard = None        # row number while touching you -> stay close to that row's poses

def _clamp(p):
    out = []
    for i, v in enumerate(p):
        sid = i + 1
        if _guard is not None and sid <= 5:
            row = ROWS[_guard]
            vals = [row["left"][i], row["right"][i], row["air"][i]]
            v = max(min(vals) - MAX_FROM_POSE, min(max(vals) + MAX_FROM_POSE, v))
        out.append(max(0, min(_LIMITS[sid], v)))
    return out

def _send(p):
    q = [int(round(v)) for v in _clamp(p)]
    arm.Arm_serial_servo_write6_array(q, STEP_MS + 30)
    if not SIM:
        time.sleep(STEP_MS / 1000)

def _quiet_read(sid):
    with contextlib.redirect_stdout(io.StringIO()):
        try:
            v = arm.Arm_serial_servo_read(sid)
        except Exception:
            v = None
    if v is None or not (0 <= v <= _LIMITS[sid]):
        return None
    return v

def where_am_i():
    pose = []
    for sid in range(1, 6):
        v = None
        for _ in range(6):
            v = _quiet_read(sid)
            if v is not None:
                break
            time.sleep(0.02)
        if v is None:
            return None
        pose.append(int(v))
    pose.append(GRIP_CLOSED)
    return pose

def follow(pose_fn, seconds, speed=1.0):
    """Follow poses pose_fn(t), t 0..1, smoothly and never faster than MAX_SPEED."""
    global _cur
    k = 400
    pts = [pose_fn(i / k) for i in range(k + 1)]
    worst = max(max(abs(a - b) for a, b in zip(p1[:5], p0[:5])) for p0, p1 in zip(pts, pts[1:]))
    gworst = max(abs(p1[5] - p0[5]) for p0, p1 in zip(pts, pts[1:]))
    seconds = max(seconds, worst * k / (MAX_SPEED * speed), gworst * k / 60, 0.2)
    n = max(1, int(seconds * 1000 / STEP_MS))
    for i in range(1, n + 1):
        _send(pose_fn(i / n))
    _cur = pose_fn(1.0)

def ease(t):
    return (1 - math.cos(math.pi * t)) / 2

def glide(target, at_least_s=0.0, speed=1.0):
    start = list(_cur)
    follow(lambda t: mix(start, target, ease(t)), at_least_s, speed)

# ---- moves (offsets around the current spot) ----------------------------------
# "L" = lift a little toward the air pose (only ever away from your back).
def ramp(t):
    return math.sin(math.pi * t) ** 0.5

def wave(t, n):
    return (1 - math.cos(2 * math.pi * n * t)) / 2

def s(t, n):
    return math.sin(2 * math.pi * n * t) * ramp(t)

def p_strokes(t):  return {1: 5 * s(t, 3)}
def p_wrist(t):    return {4: 5 * s(t, 6)}
def p_eight(t):    return {1: 5 * s(t, 2), 4: 4 * math.sin(4 * math.pi * 2 * t) * ramp(t)}
def p_claw(t):     return {6: (GRIP_OPEN - GRIP_CLOSED) * wave(t, 3), 4: 3 * s(t, 3)}
def p_zigzag(t):   return {1: 6 * math.sin(math.pi * t) * math.cos(0), 4: 3 * s(t, 10)}
def p_circles(t):  return {1: 4 * s(t, 3), 4: 4 * math.cos(2 * math.pi * 3 * t) * ramp(t)}
def p_spiral(t):
    r = math.sin(math.pi * t)
    return {1: 5 * r * math.sin(2 * math.pi * 4 * t), 4: 5 * r * math.cos(2 * math.pi * 4 * t)}
def p_knead(t):
    ph = (3 * t) % 1.0
    return {6: (GRIP_OPEN - GRIP_CLOSED) * math.sin(math.pi * ph) ** 2,
            "L": 0.08 * math.sin(math.pi * ph) ** 2, 5: 6 * s(t, 3)}
def p_tapping(t):  return {"L": 0.10 * wave(t, 8)}
def p_roll(t):     return {3: 3 * s(t, 3), 4: -4 * s(t, 3)}
def p_breathe(t):  return {1: 5 * s(t, 2), "L": 0.06 * wave(t, 4)}
def p_twist(t):    return {5: 12 * s(t, 3)}

PATTERNS = {"strokes":      (p_strokes, 7, "scratch"),
            "wrist scratch":(p_wrist,   6, "scratch"),
            "figure eight": (p_eight,   8, "scratch"),
            "claw":         (p_claw,    7, "scratch"),
            "zigzag":       (p_zigzag,  7, "scratch"),
            "circles":      (p_circles, 8, "massage"),
            "spiral":       (p_spiral, 10, "massage"),
            "kneading":     (p_knead,  10, "massage"),
            "tapping":      (p_tapping, 7, "massage"),
            "rolling wave": (p_roll,    8, "massage"),
            "breathing":    (p_breathe, 9, "massage"),
            "twist":        (p_twist,   6, "massage")}

def pick(n):
    names = [k for k, v in PATTERNS.items() if MODE == "mix" or v[2] == MODE]
    return random.sample(names, min(n, len(names)))

def do_move(name, r, side):
    fn, sec, kind = PATTERNS[name]
    base = touch(r, side)
    def pose(t):
        p = list(base)
        offs = fn(t)
        for k, v in offs.items():
            if k == "L":
                p = toward_air(p, r, v)
            else:
                p[k - 1] += v * (STRENGTH if k != 6 else 1)
        return p
    follow(pose, sec)

# ---- getting around -----------------------------------------------------------
def need_air():
    if not AIR_READY:
        raise SystemExit("Record the 'air' pose for every row first (cell 1, ROWS). The arm did not move.")

def start_safely(first_row=0):
    global _cur, _guard
    need_air()
    _guard = None
    pose = where_am_i()
    if pose is None:
        raise SystemExit("Can't read the arm position - check USB, restart the kernel. (The arm did not move.)")
    _cur = pose
    print("Arm is at", pose[:5], "- gliding slowly to the start (through the air)...")
    glide(air(first_row), at_least_s=4)

def land(r, side):
    """From the row's air pose: come down slowly onto your back."""
    global _guard
    glide(hover(r, side), at_least_s=2)
    _guard = r
    glide(touch(r, side), at_least_s=2, speed=0.6)

def take_off(r):
    global _guard
    up = toward_air(list(_cur), r, 0.35)
    glide(up, at_least_s=1)
    _guard = None
    glide(air(r), at_least_s=1.5)

def slide(r, a, b, seconds=7):
    """Slide across the row (left <-> right), lightly lifted in the middle, small zigzags."""
    def pose(t):
        side = a + (b - a) * ease(t)
        p = touch(r, side)
        p = toward_air(p, r, 0.12 * math.sin(math.pi * t))   # never presses in between the points
        p[3] += 3 * s(t, 4) * STRENGTH
        return p
    follow(pose, seconds, 0.7)

def change_row(r_from, r_to):
    """Lift off, travel through the air poses, never across your body."""
    take_off(r_from)
    glide(air(r_to), at_least_s=3)

def go_to_sleep():
    global _guard
    if _cur is None:
        return
    if _guard is not None:
        take_off(_guard)
    if SLEEP:
        print("Going to sleep...")
        glide(SLEEP, at_least_s=3)
    print("Sleeping.")

# ---- the cells call these -----------------------------------------------------
def read_angles():
    """Motors go LOOSE - HOLD THE ARM. Prints the 6 numbers every second. Stop button to end."""
    print("HOLD THE ARM - motors go loose in 3 s!")
    time.sleep(0 if SIM else 3)
    arm.Arm_serial_set_torque(0)
    try:
        while True:
            p = where_am_i()
            print(p if p else "(no reading)")
            if SIM:
                break
            time.sleep(1)
    except KeyboardInterrupt:
        pass
    finally:
        arm.Arm_serial_set_torque(1)
        print("Motors on again.")

def check_path():
    """NOBODY on the bed. Visits every air pose, and hovers (35% lifted) over every spot."""
    try:
        start_safely(0)
        for r in range(len(ROWS)):
            if r > 0:
                print(f"  through the air to row '{ROWS[r]['name']}'")
                glide(air(r), at_least_s=3)
            for side, name in ((0, "left"), (1, "right")):
                print(f"    hovering over {ROWS[r]['name']} {name}")
                glide(hover(r, side), at_least_s=2)
                time.sleep(0 if SIM else 1)
            print("    sliding across the row (in the air)")
            follow(lambda t: hover(r, 1 - ease(t)), 6, 0.7)
            glide(air(r), at_least_s=1.5)
        print("If nothing came near your head/neck area and nothing hit the bed: set PATH_CHECKED = True.")
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        go_to_sleep()

def demo_in_air():
    """Every move, in the air above the top row."""
    global _guard
    try:
        start_safely(0)
        glide(hover(0, 0.5), at_least_s=2)
        for name in PATTERNS:
            print("  move:", name)
            fn, sec, kind = PATTERNS[name]
            base = list(_cur)
            def pose(t, fn=fn, base=base):
                p = list(base)
                for k, v in fn(t).items():
                    if k == "L":
                        p = toward_air(p, 0, v)
                    else:
                        p[k - 1] += v * (STRENGTH if k != 6 else 1)
                return p
            follow(pose, sec)
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        glide(air(0), at_least_s=1.5)
        go_to_sleep()

def scratch():
    need_air()
    if not PATH_CHECKED:
        raise SystemExit("First run 'Step 1: check the path' (nobody on the bed), then set PATH_CHECKED = True.")
    end = time.time() + MINUTES * 60
    left = lambda: end - time.time()
    try:
        start_safely(0)
        print(f"Session: {MINUTES} min, mode '{MODE}'. Stop button = end.")
        order = list(range(len(ROWS)))
        rnd = 0
        r = order[0]
        land(r, 0)
        print(f"  warm-up: slow strokes across the {ROWS[r]['name']} row")
        slide(r, 0, 1, 9); slide(r, 1, 0, 9)
        side = 0
        while left() > 30:
            rows = order if rnd % 2 == 0 else order[::-1]      # top -> bottom, then back up
            for rr in rows:
                if left() <= 30:
                    break
                if rr != r:
                    print(f"  -> through the air to the {ROWS[rr]['name']} row")
                    change_row(r, rr)
                    r = rr
                    land(r, side)
                for target in (side, 0.5, 1 - side):            # one side, middle, other side
                    if left() <= 30:
                        break
                    if target != side:
                        slide(r, side, target, 4)
                        side = target
                    for name in pick(2):
                        if left() <= 30:
                            break
                        print(f"     {ROWS[r]['name']} {['left','middle','right'][int(side*2)]}: {name}")
                        do_move(name, r, side)
            rnd += 1
            if SIM and rnd >= 2:
                break
        print("  finish: gentle tapping and one calm stroke")
        do_move("tapping", r, side)
        slide(r, side, 1 - side, 9)
        print("Done.")
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        go_to_sleep()

arm = connect()
print("Connected" if not SIM else "Simulation mode")
print("Rows:", [r["name"] for r in ROWS], "| air poses recorded:", AIR_READY)


## Read angles (for the air poses)
⚠️ Motors go loose - **hold the arm**. For each row: put the gripper on that row, then lift it 5-10 cm straight up away from your back/pillow and copy the numbers into that row's `"air"`. ■ Stop to end.

In [ ]:
read_angles()

## Step 1: check the path (NOBODY on the bed)
Visits every air pose and hovers over every spot. Watch that it never comes near the head area and never swings into the bed. Then set `PATH_CHECKED = True` and run cells 1 and 2 again.

In [ ]:
check_path()

## Step 2: all moves in the air (optional)

In [ ]:
demo_in_air()

## Step 3: Massage!
Try with your hand first, then your back.

In [ ]:
scratch()

## Sleep now

In [ ]:
go_to_sleep()

## Voice control (Google Home)
Run once after cells 1 and 2, then **Listen**. ■ Stop ends listening and parks the arm.

In [ ]:
import threading, urllib.request, json as _json

# Your private "mailbox" name on ntfy.sh. Must be EXACTLY the same in the IFTTT web request URL.
# Keep it secret - anyone who knows it could start the arm.
NTFY_TOPIC = "backarm-4d4e72eb9e"

_abort = False
_worker = None
_orig_send = _send

def _send(p):                       # every small step checks for a "stop"
    global _abort
    if _abort:
        _abort = False
        raise KeyboardInterrupt
    _orig_send(p)

def _run(mode):
    global MODE
    MODE = mode
    try:
        scratch()
    except SystemExit as e:
        print("Not started:", e)

def busy():
    return _worker is not None and _worker.is_alive()

def start(mode):
    global _worker
    if busy():
        print("Already running - say 'stop' first.")
        return
    _worker = threading.Thread(target=_run, args=(mode,), daemon=True)
    _worker.start()

def stop():
    global _abort
    if busy():
        _abort = True

def handle(word):
    global MAX_SPEED
    print(f"[voice] '{word}'")
    if "stop" in word or "sleep" in word:
        stop()
    elif "massage" in word or "massege" in word:
        start("massage")
    elif "scratch" in word:
        start("scratch")
    elif "mix" in word or "back" in word:
        start("mix")
    elif "slow" in word:
        MAX_SPEED = max(10, MAX_SPEED * 0.75); print("  speed now", round(MAX_SPEED))
    elif "fast" in word:
        MAX_SPEED = min(40, MAX_SPEED * 1.25); print("  speed now", round(MAX_SPEED))
    else:
        print("  (unknown command)")

def listen():
    url = f"https://ntfy.sh/{NTFY_TOPIC}/json"
    print("Listening for voice commands... (press Stop button to quit)")
    try:
        while True:
            try:
                with urllib.request.urlopen(url, timeout=90) as r:
                    for line in r:
                        msg = _json.loads(line)
                        if msg.get("event") == "message":
                            handle(msg.get("message", "").strip().lower())
            except KeyboardInterrupt:
                raise
            except Exception as e:
                print("  connection problem, retrying in 5 s:", e)
                time.sleep(5)
    except KeyboardInterrupt:
        print("Quitting listener - stopping the arm.")
        stop()
        if _worker:
            _worker.join(timeout=60)

print("Voice ready. Topic:", NTFY_TOPIC)


## Listen

In [ ]:
listen()